# Contract Auditor — Colab Prototype

Interactive scratchpad for the same pipeline used in `app.py`:
PDF -> text extraction -> chunking -> embeddings -> FAISS -> Groq answer + validation.

**Before running:** Runtime -> Change runtime type -> T4 GPU -> Save.
Add your Groq key under the key icon in the left sidebar as a secret named `GROQ_API_KEY`.

In [ ]:
!pip install -q pypdf langchain-text-splitters sentence-transformers faiss-cpu groq

## 1. Upload a sample PDF

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick a contract/policy PDF from your machine
pdf_filename = list(uploaded.keys())[0]
print('Uploaded:', pdf_filename)

## 2. Extract text page-by-page with pypdf

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_filename)
pages = []
for i, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ''
    if text.strip():
        pages.append((i, text))

print(f'Extracted text from {len(pages)} page(s).')
print(pages[0][1][:500] if pages else 'No text found.')

## 3. Chunk with RecursiveCharacterTextSplitter (1000 chars, 200 overlap)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=['\n\n', '\n', '. ', ' ', '']
)

chunks = []  # list of dicts: text, page
for page_number, page_text in pages:
    for c in splitter.split_text(page_text):
        chunks.append({'text': c, 'page': page_number})

print(f'Created {len(chunks)} chunks.')
print(chunks[0]['text'][:300])

## 4. Embed chunks and build a FAISS index

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')  # runs on GPU automatically if available

texts = [c['text'] for c in chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True, convert_to_numpy=True).astype('float32')

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print('FAISS index size:', index.ntotal)

## 5. Try a similarity search

In [ ]:
query = 'What is the termination notice period?'
query_vec = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')

k = 5
distances, indices = index.search(query_vec, k)
top_chunks = [chunks[i] for i in indices[0]]

for tc in top_chunks:
    print(f"--- page {tc['page']} ---")
    print(tc['text'][:300])
    print()

## 6. Generate an answer with Groq, then run the Corrective RAG validation pass

In [ ]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
context = '\n\n---\n\n'.join(f"[Page {tc['page']}]\n{tc['text']}" for tc in top_chunks)

answer_resp = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[
        {'role': 'system', 'content': 'Answer ONLY from the given context. Cite the page number for each fact.'},
        {'role': 'user', 'content': f'CONTEXT:\n{context}\n\nQUESTION:\n{query}'}
    ],
    temperature=0.1,
)
draft_answer = answer_resp.choices[0].message.content
print('DRAFT ANSWER:\n', draft_answer)

In [ ]:
validation_resp = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[
        {'role': 'system', 'content': 'You are a strict fact-checker. Verify the draft answer is fully supported by the context. Reply with VERDICT: SUPPORTED|PARTIALLY_SUPPORTED|NOT_SUPPORTED and explain why.'},
        {'role': 'user', 'content': f'CONTEXT:\n{context}\n\nQUESTION:\n{query}\n\nDRAFT ANSWER:\n{draft_answer}'}
    ],
    temperature=0.0,
)
print('VALIDATION:\n', validation_resp.choices[0].message.content)

## Next step
Once this all looks right for your test documents, move to running the full app locally:
`streamlit run app.py` (see the main README.md for setup).